In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 📚 Research Librarian

## What We're Building

A RAG assistant that searches across **multiple research papers** and returns
answers with **paper-by-paper evidence** — like a research librarian that pulls
relevant passages from different papers and tells you exactly which paper said
what.

## Why Paper-by-Paper Evidence Matters

When reviewing literature you don't want a blended answer. You want:
- "Paper A found X (p. 3)"
- "Paper B contradicts this, finding Y (p. 7)"
- "Paper C supports A with additional evidence (Abstract)"

This lets you evaluate the strength of evidence yourself.

## Stack
- **LangChain** — orchestration + retrieval
- **ChromaDB** — vector store with per-paper metadata
- **Ollama** — local LLM + embeddings
- **Multi-source citation** — group and display evidence by paper

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

from collections import defaultdict
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Sample Research Papers

Three simulated ML research papers with distinct findings. In production
you'd load PDFs from a folder.

In [ ]:
papers = {
    "Chen et al. 2024 — Scaling Laws for RAG": {
        "authors": "Wei Chen, Sarah Liu, James Park",
        "year": 2024,
        "venue": "NeurIPS",
        "content": """ABSTRACT

We study scaling laws for retrieval-augmented generation systems. By varying
the number of retrieved documents (k=1 to k=50), corpus size (10K to 10M
documents), and model size (1B to 70B parameters), we identify three regimes
of RAG performance.

1. INTRODUCTION

Retrieval-Augmented Generation (RAG) has become the standard approach for
grounding large language models in external knowledge. However, the
interaction between retrieval quality, context length, and model capacity
is poorly understood. Prior work has shown that simply increasing k does
not always improve performance, and may even degrade it for smaller models.

2. METHODOLOGY

We benchmark on three tasks: open-domain QA (Natural Questions), fact
verification (FEVER), and multi-hop reasoning (HotpotQA). We use Contriever
for retrieval and vary the reader model across LLaMA-2 7B, 13B, and 70B.

3. KEY FINDINGS

Finding 1: There exists an optimal k for each model size. For 7B models,
performance peaks at k=5 and degrades beyond k=10 due to context confusion.
For 70B models, performance improves monotonically up to k=20.

Finding 2: Corpus quality matters more than corpus size. A curated 100K
corpus outperforms a noisy 1M corpus by 8 accuracy points on NQ.

Finding 3: Chunk size interacts with k. Smaller chunks (200 tokens) with
higher k outperform larger chunks (1000 tokens) with lower k, because
smaller chunks provide more precise evidence.

4. CONCLUSION

RAG systems should be tuned jointly across k, chunk size, and model capacity.
We release scaling curves to help practitioners select optimal configurations.
""",
    },
    "Martinez & Wong 2024 — Hybrid Retrieval Beats Dense": {
        "authors": "Elena Martinez, Kevin Wong",
        "year": 2024,
        "venue": "ACL",
        "content": """ABSTRACT

We demonstrate that hybrid retrieval (combining BM25 keyword search with
dense embeddings) consistently outperforms either method alone across six
benchmark datasets. The improvement is especially significant for queries
containing rare technical terms.

1. INTRODUCTION

Dense retrieval using learned embeddings has become dominant, but it has
known failure modes: rare terms, exact string matching, and numerical
queries. BM25 excels in these areas but lacks semantic understanding.
We propose a simple yet effective fusion strategy.

2. APPROACH

We use Reciprocal Rank Fusion (RRF) to combine BM25 and dense retriever
ranked lists. For each document d, the fused score is:
  score(d) = sum( 1 / (k + rank_i(d)) ) for each retriever i
where k=60 is a smoothing constant.

3. RESULTS

On Natural Questions: BM25 alone = 42.1, Dense alone = 48.3, Hybrid = 52.7
On TechQA (domain-specific): BM25 = 51.2, Dense = 44.8, Hybrid = 55.9
On FEVER: BM25 = 71.0, Dense = 78.3, Hybrid = 80.1

Key insight: Hybrid retrieval shows the largest gains on domain-specific
datasets where technical vocabulary is common. On general QA, the gains
are modest but consistent.

4. COMPARISON WITH RERANKING

We also tested using a cross-encoder reranker on top of dense retrieval.
Reranking improves dense-only from 48.3 to 51.0 on NQ, which is still
below our hybrid approach (52.7). However, combining hybrid retrieval
with reranking achieves the best result: 54.2.

5. CONCLUSION

Hybrid retrieval should be the default for production RAG systems.
The computational overhead of running BM25 in parallel is negligible
compared to the quality gains.
""",
    },
    "Park et al. 2023 — When RAG Fails": {
        "authors": "David Park, Nina Thompson, Raj Patel",
        "year": 2023,
        "venue": "EMNLP",
        "content": """ABSTRACT

We systematically study failure modes of RAG systems. We categorize failures
into retrieval failures (relevant document not found), context failures
(relevant document found but answer not extracted), and integration failures
(answer extracted but combined incorrectly with model knowledge).

1. INTRODUCTION

RAG has become widely adopted, but failure analysis remains superficial.
Most evaluations report only end-to-end accuracy without diagnosing where
failures occur in the pipeline.

2. FAILURE TAXONOMY

Type 1 — Retrieval Failure (38% of errors): The retriever does not return
any relevant document in the top-k. This is most common for multi-hop
questions requiring bridging concepts.

Type 2 — Context Failure (31% of errors): The relevant document is
retrieved, but the answer span is not in the selected chunk due to poor
chunking. Tables and lists are especially problematic.

Type 3 — Integration Failure (24% of errors): The model has the right
context but produces an incorrect answer, often blending retrieved facts
with its own (wrong) parametric knowledge.

Type 4 — Hallucination Despite Context (7% of errors): The model ignores
retrieved context entirely and generates a plausible-sounding but wrong
answer from parametric memory.

3. MITIGATION STRATEGIES

For Type 1: Query expansion and multi-step retrieval reduce failures by 45%.
For Type 2: Table-aware chunking and overlapping chunks reduce failures by 60%.
For Type 3: Adding "Answer ONLY from the provided context" to prompts
reduces integration failures by 35%.
For Type 4: Confidence calibration and abstention training help, but remain
challenging — only 15% reduction observed.

4. CONCLUSION

The majority of RAG failures are addressable with engineering improvements.
Retrieval and chunking issues (Types 1+2) account for 69% of all errors
and have effective mitigations. Integration and hallucination (Types 3+4)
require model-level solutions and remain open problems.
""",
    },
}

print(f"Loaded {len(papers)} papers:")
for title, meta in papers.items():
    print(f"  • {title} ({meta['venue']} {meta['year']})")

## Step 3 — Index Papers with Rich Metadata

In [ ]:
def chunk_paper(title: str, meta: dict) -> list[Document]:
    """Split a paper into chunks with metadata for every chunk."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=80,
        separators=["\n\n", "\n", ". "],
    )
    
    text = meta["content"]
    # Detect section headers (lines like "1. INTRODUCTION")
    import re
    sections = re.split(r'\n(?=\d+\.\s+[A-Z])', text)
    
    docs = []
    for section in sections:
        # Extract section name from first line
        first_line = section.strip().split("\n")[0]
        section_name = first_line.strip()
        
        section_doc = Document(
            page_content=section.strip(),
            metadata={
                "paper_title": title,
                "authors": meta["authors"],
                "year": meta["year"],
                "venue": meta["venue"],
                "section": section_name,
            },
        )
        split_docs = splitter.split_documents([section_doc])
        docs.extend(split_docs)
    
    return docs


all_chunks = []
for title, meta in papers.items():
    chunks = chunk_paper(title, meta)
    all_chunks.extend(chunks)
    print(f"  {title}: {len(chunks)} chunks")

print(f"\nTotal: {len(all_chunks)} chunks")

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="research_librarian",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

print(f"Indexed {vectorstore._collection.count()} chunks from {len(papers)} papers")

## Step 4 — Paper-by-Paper Evidence Display

The core of this project: group retrieved chunks by paper and show
evidence with full citations.

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

librarian_prompt = ChatPromptTemplate.from_template(
    """You are a research librarian assistant. Answer the question using evidence
from the retrieved research papers.

IMPORTANT GUIDELINES:
- For each claim, cite the specific paper (Author et al., Year)
- Note when papers agree or contradict each other
- Distinguish between findings, methodology, and speculation
- If evidence is insufficient, say so explicitly
- End with a brief synthesis across papers

Retrieved Evidence:
{context}

Question: {question}

Literature Review Answer:"""
)


def ask_librarian(question: str) -> None:
    """Query the research library and get paper-by-paper evidence."""
    print(f"🔍 {question}")
    print("=" * 60)
    
    # Retrieve
    docs = retriever.invoke(question)
    
    # Group by paper for display
    by_paper = defaultdict(list)
    for doc in docs:
        key = doc.metadata["paper_title"]
        by_paper[key].append(doc)
    
    # Show evidence per paper
    print(f"\n📚 Evidence from {len(by_paper)} papers:\n")
    context_parts = []
    for paper_title, chunks in by_paper.items():
        meta = chunks[0].metadata
        print(f"📄 {paper_title}")
        print(f"   Authors: {meta['authors']} | Venue: {meta['venue']} {meta['year']}")
        print(f"   Sections referenced: {', '.join(set(c.metadata['section'][:40] for c in chunks))}")
        print(f"   Chunks retrieved: {len(chunks)}")
        print()
        
        for chunk in chunks:
            context_parts.append(
                f"[{paper_title} | {meta['authors']} | "
                f"{meta['venue']} {meta['year']} | "
                f"Section: {chunk.metadata['section'][:40]}]\n"
                f"{chunk.page_content}"
            )
    
    # Generate answer
    context = "\n\n---\n\n".join(context_parts)
    chain = librarian_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    
    print("-" * 60)
    print(f"\n{answer}")
    print("\n" + "=" * 60)


print("Research librarian ready!")

## Step 5 — Query the Literature

In [ ]:
ask_librarian("What is the optimal number of retrieved documents (k) for RAG?")

In [ ]:
ask_librarian("How does hybrid retrieval compare to dense-only retrieval?")

In [ ]:
ask_librarian("What are the main failure modes of RAG and how can they be mitigated?")

In [ ]:
# Cross-paper question — requires evidence synthesis
ask_librarian(
    "What do the papers say about chunk size? Is there consensus "
    "across different studies?"
)

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Per-paper metadata** | Every chunk knows its paper, authors, venue, section |
| **Section-aware chunking** | Splits by paper sections first, then by size |
| **Evidence grouping** | Groups retrieved chunks by paper for clear citations |
| **Cross-paper synthesis** | LLM compares/contrasts findings across papers |
| **Rich retrieval display** | Shows which papers and sections matched before the answer |

## 🔧 Customization Ideas

- **PDF loader**: Use `pypdf` or `pymupdf` to load real papers from a folder
- **Citation graph**: Track which papers cite each other
- **Evidence strength**: Rank papers by relevance score and recency
- **Export**: Generate a bibliography in BibTeX format from retrieved papers